In [ ]:
import pandas as pd
import xgboost as xgb

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [ ]:
df = pd.read_csv('../001.preprocess/train/train_sort_A.csv')

In [ ]:
# カテゴリカルデータを数値に変換
user_encoder = LabelEncoder()
product_encoder = LabelEncoder()

df['user_id'] = user_encoder.fit_transform(df['user_id'])
df['product_id'] = product_encoder.fit_transform(df['product_id'])

In [ ]:
# 使用する特徴量
features = ["user_id", "product_id", "event_type", "ad", "user_interest", "product_interest", "hour", "dayofweek"]
target = "product_interest"

X = df[features]
y = df[target]

In [ ]:
# --- XGBoost 用のデータセット作成 ---
# ユーザーごとのクエリグループを定義
query_group = df.groupby("user_id").size().tolist()

In [ ]:
#ユーザーごとに分割
unique_users = df["user_id"].unique()
train_users, valid_users = train_test_split(unique_users, test_size=0.2, random_state=42)

In [ ]:
#各データをユーザーごとに分割
train_df = df[df["user_id"].isin(train_users)]
valid_df = df[df["user_id"].isin(valid_users)]

In [ ]:
#特徴量とターゲットを設定
X_train, y_train = train_df[features], train_df[target]
X_valid, y_valid = valid_df[features], valid_df[target]

In [ ]:
#クエリグループを作成
train_group = train_df.groupby("user_id").size().tolist()
valid_group = valid_df.groupby("user_id").size().tolist()

In [ ]:
#XGBoost 用データセット
dtrain = xgb.DMatrix(X_train, label=y_train)
dvalid = xgb.DMatrix(X_valid, label=y_valid)

In [ ]:
#クエリグループを設定
dtrain.set_group(train_group)
dvalid.set_group(valid_group)

In [ ]:
# --- XGBoost モデルの訓練 ---
params = {
    "objective": "rank:ndcg",  # NDCG 最適化
    "eval_metric": "ndcg",
    "eta": 0.1,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "tree_method": "hist",
}

model = xgb.train(params, dtrain, num_boost_round=100, evals=[(dvalid, "valid")], early_stopping_rounds=10, verbose_eval=10)

In [ ]:
# --- 予測とランキング ---
# 評価データの読み込み
eval_df = pd.read_csv("eval_data.csv")  # user_id のみを含む

In [ ]:
# 評価用データに対して全商品の組み合わせを作成
all_products = df["product_id"].unique()
eval_users = eval_df["user_id"].unique()

test_data = []
for user in eval_users:
    for product in all_products:
        test_data.append([user, product])

test_df = pd.DataFrame(test_data, columns=["user_id", "product_id"])

In [ ]:
# 必要な特徴量を追加
test_df["event_type"] = 1  # 仮の値
test_df["ad"] = -1
test_df["user_interest"] = df["user_interest"].mean()  # 平均値で埋める
test_df["product_interest"] = df["product_interest"].mean()  # 平均値で埋める
test_df["hour"] = 12  # 仮の値
test_df["dayofweek"] = 3  # 仮の値

In [ ]:
# カテゴリ変換
test_df["user_id"] = user_encoder.transform(test_df["user_id"])
test_df["product_id"] = product_encoder.transform(test_df["product_id"])

In [ ]:
# 予測
dtest = xgb.DMatrix(test_df[features])
test_df["score"] = model.predict(dtest)

In [ ]:
# --- ユーザーごとにランキング ---
test_df["rank"] = test_df.groupby("user_id")["score"].rank(method="first", ascending=False)

In [ ]:
# 商品IDを元のフォーマットに戻す
test_df["user_id"] = user_encoder.inverse_transform(test_df["user_id"])
test_df["product_id"] = product_encoder.inverse_transform(test_df["product_id"])

In [ ]:
# 結果の保存
test_df[["user_id", "product_id", "rank"]].to_csv("./predict/predictions.csv", index=False, header=False, sep="\t")

print("予測完了！結果は predictions.csv に保存されました。")
